In [1]:
# Import the Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load dataset
df = pd.read_csv('American Express User Exit Prediction.csv')
df

,Credit Score,Geography,Gender,Age,Customer Since,Current Account,Num of products,UPI Enabled,Estimated Yearly Income,Closed
0,553,Delhi,Female,45,4,0.000000e+00,4,1,274150,0
1,447,Bengaluru,Male,31,7,0.000000e+00,4,1,519360,0
2,501,Delhi,Female,32,2,0.000000e+00,4,1,545501,0
3,428,Delhi,Male,51,3,0.000000e+00,4,1,86868,0
4,492,Delhi,Female,57,6,1.912682e+06,2,1,518680,0
...,...,...,...,...,...,...,...,...,...,...
9922,594,Bengaluru,Male,28,6,0.000000e+00,4,1,394810,0
9923,557,Bengaluru,Male,59,3,8.050490e+05,2,0,58163,1
9924,627,Mumbai,Female,42,4,1.893594e+06,4,0,494067,0
9925,600,Bengaluru,Female,51,0,9.031778e+05,2,1,109375,1


In [3]:
# for checking random sample
df.sample(1)

,Credit Score,Geography,Gender,Age,Customer Since,Current Account,Num of products,UPI Enabled,Estimated Yearly Income,Closed
5548,550,Bengaluru,Female,34,3,887073.7684,2,1,341870,0


In [4]:
# Data Preprocessing
df = pd.get_dummies(df, columns=['Geography', 'Gender'], drop_first=True, dtype=int)
df.loc[df['Age'] > 100, 'Age'] = 100
df['Current Account'] = df['Current Account'].astype(int)

In [5]:
df.sample(1)

,Credit Score,Age,Customer Since,Current Account,Num of products,UPI Enabled,Estimated Yearly Income,Closed,Geography_Delhi,Geography_Mumbai,Gender_Male
3235,556,39,4,1953193,2,0,73419,0,0,1,0


In [6]:
# Split Dataset into train & Test
X = df.drop(['Closed'], axis=1)
y = df['Closed']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [7]:
# Feature Scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## If Required

In [8]:
# Handling Class Imbalance with SMOTE
from imblearn.over_sampling import SMOTE

print("Original class distribution:")
print(y_train.value_counts())

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print("\nClass distribution after SMOTE:")
print(y_train_res.value_counts())

Original class distribution:
Closed
0    5530
1    1418
Name: count, dtype: int64

Class distribution after SMOTE:
Closed
0    5530
1    5530
Name: count, dtype: int64


## Implement Artificial Neural Network

In [9]:
import tensorflow as tf
ANN = tf.keras.models.Sequential()
ANN.add(tf.keras.layers.Dense(units=8, activation='relu'))
ANN.add(tf.keras.layers.Dense(units=8, activation='relu'))
ANN.add(tf.keras.layers.Dense(units=8, activation='relu')) # Added an additional hidden layer
ANN.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

In [10]:
ANN.build(input_shape=(None, X_train_res.shape[1])) # Build the model explicitly with the correct input shape
ANN.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 8)              │            88 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 241 (964.00 B)

 Trainable params: 241 (964.00 B)

 Non-trainable params: 0 (0.00 B)

Best parameters found:  {'batch_size': 64, 'epochs': 137, 'model__units': 8, 'optimizer': 'rmsprop'}

In [11]:
ANN.compile(optimizer = 'rmsprop', loss = 'binary_crossentropy', metrics = ['accuracy'])
ANN.fit(X_train_res, y_train_res, batch_size = 64, epochs = 137)

Epoch 1/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5560 - loss: 0.6835
Epoch 2/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6342 - loss: 0.6461
Epoch 3/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6724 - loss: 0.6158
Epoch 4/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6958 - loss: 0.5932
Epoch 5/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7091 - loss: 0.5768
Epoch 6/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7163 - loss: 0.5663
Epoch 7/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7242 - loss: 0.5585
Epoch 8/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7297 - loss: 0.5516
Epoch 9/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7308 - loss: 0.5463
Epoch 10/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7324 - loss: 0.5411
Epoch 11/137
173/173 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7384 - loss: 0.5360
Epoch 12/137
173/173 ━━━━━━━━━━━━━━━━━━━━

In [12]:
y_pred = ANN.predict(X_test)
y_pred = (y_pred > 0.68)
# print(np.concatenate((y_pred.reshape(len(y_pred),1), y_test.reshape(len(y_test),1)),1))

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [13]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score

# Calculate and print the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Generate and print the classification report
print(classification_report(y_test, y_pred))

# Calculate and print the ROC-AUC score
roc_auc = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC Score: {roc_auc*100:.2f}%")

Accuracy: 0.82
              precision    recall  f1-score   support

           0       0.90      0.87      0.88      2369
           1       0.55      0.60      0.58       610

    accuracy                           0.82      2979
   macro avg       0.72      0.74      0.73      2979
weighted avg       0.82      0.82      0.82      2979

ROC-AUC Score: 73.87%


In [14]:
import joblib

# Save the trained ANN model
ANN.save('churn_model.h5')

# Save the scaler (Crucial: the web app needs the EXACT same scaling parameters)
joblib.dump(sc, 'scaler.joblib')

# Save the list of feature names to ensure order
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'features.joblib')

['features.joblib']

In [15]:
import tensorflow as tf

# Load the model from the .h5 format
loaded_model = tf.keras.models.load_model('churn_model.h5')

# Save the model in the recommended .keras format
loaded_model.save('churn_model.keras')
print("Model converted and saved as 'churn_model.keras'")

Model converted and saved as 'churn_model.keras'
